# 01: Classical Time Series Forecasting: ARIMA, SARIMAX & Stationarity Tests

**Track 07: Statistical Time-Series Forecasting** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Comprehensive time-series econometrics: Augmented Dickey-Fuller (ADF) stationarity testing, differencing ($d$), ACF/PACF auto-correlation analysis, and SARIMAX model forecasting.


## 1. Time Series Stationarity & Augmented Dickey-Fuller (ADF) Test
A stationary series has constant mean, constant variance, and autocovariance independent of time.

ADF Hypothesis:
$H_0$: Series possesses a unit root (non-stationary)
$H_1$: Series is stationary

In [ ]:
import os
import sys
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / "utils").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

from utils.data_loader import load_dataset

import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose

df = load_dataset("monthly_sales")
date_col = "Date" if "Date" in df.columns else ("Month" if "Month" in df.columns else df.columns[0])
target_col = "Sales" if "Sales" in df.columns else df.columns[-1]

df[date_col] = pd.to_datetime(df[date_col])
df = df.set_index(date_col).sort_index()

sales = df[target_col]
print(f"Sales Records: {len(sales)} periods")

# Run ADF Test
adf_stat, p_value, lags, nobs, crit_vals, icbest = adfuller(sales)
print("=== Augmented Dickey-Fuller (ADF) Stationarity Test ===")
print(f"ADF Statistic: {adf_stat:.4f}")
print(f"p-value      : {p_value:.4f}")
print(f"Stationary at 5% level? {p_value < 0.05}")


## 2. Differencing & SARIMAX Fitting
Fit SARIMAX $(p=1, d=1, q=1) \times (P=1, D=1, Q=0)_{12}$ on monthly sales data.

In [ ]:
train_sales = sales.iloc[:-6]
test_sales = sales.iloc[-6:]

model = SARIMAX(train_sales, order=(1, 1, 1), seasonal_order=(1, 1, 0, 12), enforce_stationarity=False, enforce_invertibility=False)
sarimax_fit = model.fit(disp=False)

forecast = sarimax_fit.forecast(steps=6)
mae = np.mean(np.abs(test_sales.values - forecast.values))
mape = np.mean(np.abs((test_sales.values - forecast.values) / test_sales.values)) * 100

print("=== SARIMAX Forecast Results (6 Months Horizon) ===")
print(pd.DataFrame({"Actual": test_sales.values, "Forecast": forecast.values.round(2)}, index=test_sales.index))
print(f"\nForecast MAE : ${mae:.2f}")
print(f"Forecast MAPE: {mape:.2f}%")